In [1]:
'''
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/transformers/projekt/tf_project/src')
print(f"Töökataloog: {os.getcwd()}")
'''

!pip install --upgrade vllm
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from transformers import AutoTokenizer
import torch
from openai import OpenAI

import json
import csv
import re
import math
import numpy as np

from pydantic import BaseModel
from enum import Enum
# from huggingface_hub import login
# HF_TOKEN = ""
# login(HF_TOKEN)  # This logs you in for the session. This is needed for some specific models that require some consent.

# import requests
# response = requests.get("https://huggingface.co")
# print(response.status_code)  # Should print 200 if the connection is successful

model_name = "neuralmagic/Meta-Llama-3.1-70B-Instruct-quantized.w4a16"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

llm = LLM(model=model_name, device=device, max_model_len=16384, tensor_parallel_size=1, enable_prefix_caching=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained('neuralmagic/Meta-Llama-3.1-70B-Instruct-quantized.w4a16')


sampling_params = SamplingParams(temperature=0, max_tokens=8192)
# sampling_params = SamplingParams(temperature=0.5, max_tokens=1000)

# Structure the messages list
messages = [
    # System prompt
    {"role": "system", "content": "You are a highly intelligent conversation bot."},
    # User prompt
    {"role": "user", "content": "What do you know about Sven Laur?"}
]

prompts = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

# Use messages as input (if your model supports chat-like input)
outputs = llm.generate(
    prompts=prompts,  # Pass messages instead of a raw string
    sampling_params=sampling_params,
)

print(outputs[0].outputs[0].text)




INFO 12-20 14:24:27 gptq_marlin.py:107] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
WARNING 12-20 14:24:27 config.py:380] Async output processing is only supported for CUDA, TPU, XPU. Disabling it for other platforms.
INFO 12-20 14:24:27 llm_engine.py:237] Initializing an LLM engine (v0.6.3.post1) with config: model='neuralmagic/Meta-Llama-3.1-70B-Instruct-quantized.w4a16', speculative_config=None, tokenizer='neuralmagic/Meta-Llama-3.1-70B-Instruct-quantized.w4a16', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=gptq_marlin, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=Decoding

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 12-20 14:24:52 model_runner.py:1067] Loading model weights took 36.8637 GB
INFO 12-20 14:25:04 gpu_executor.py:122] # GPU blocks: 5974, # CPU blocks: 819
INFO 12-20 14:25:04 gpu_executor.py:126] Maximum concurrency for 16384 tokens per request: 5.83x
INFO 12-20 14:25:07 model_runner.py:1395] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 12-20 14:25:07 model_runner.py:1399] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 12-20 14:25:25 model_runner.py:1523] Graph capturing finished in 18 secs.


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.57it/s, est. speed input: 83.50 toks/s, output: 26.78 toks/s]

Sven Laur is an Estonian sailor who has competed in the Finn class.


In [5]:
#!pip uninstall -y config
import re
import pandas as pd
import numpy as np
import os
from model_llama import Model

#script_dir = os.path.dirname(os.path.abspath(__file__))
#base_path =script_dir+"/../datasets/"
base_path = "../datasets/"

# 3. For strategyqa_dataset:
strategyqa_dev_df = pd.read_json(os.path.join(base_path, 'strategyqa_dataset', 'dev.json'))
strategyqa_train_df = pd.read_json(os.path.join(base_path, 'strategyqa_dataset', 'train.json'))

# 4. For gsm8k_datasets:
gsm8k_test_df = pd.read_parquet(os.path.join(base_path, 'gsm8k_datasets', 'test-00000-of-00001.parquet'))
gsm8k_train_df = pd.read_parquet(os.path.join(base_path, 'gsm8k_datasets', 'train-00000-of-00001.parquet'))

# 5. For maqa_datasets:
maqa_dfs = {}
maqa_dataset_path = os.path.join(base_path, 'maqa_datasets')
for filename in os.listdir(maqa_dataset_path):
    if filename.endswith('.json'):
        file_path = os.path.join(maqa_dataset_path, filename)
        df_name = os.path.splitext(filename)[0] # Get filename without extension
        maqa_dfs[df_name] = pd.read_json(file_path)

# Set display option to show full column content
pd.set_option('display.max_colwidth', None)

all_datasets = {
    'strategyqa_dev': strategyqa_dev_df,
    'strategyqa_train': strategyqa_train_df,
    'gsm8k_test': gsm8k_test_df,
    'gsm8k_train': gsm8k_train_df
}

# Add MAQA datasets to the main dictionary
for name, df in maqa_dfs.items():
    all_datasets[f'maqa_{name}'] = df


def get_processed_answer_list(answer_input):

    if isinstance(answer_input, list):
        # If it's already a list, ensure all items are strings
        return [str(item) for item in answer_input]
    elif isinstance(answer_input, (bool, int, float)):
        # Convert booleans, ints, floats to string and wrap in a list
        return [str(answer_input)]
    elif isinstance(answer_input, str):
        answer_string = answer_input

        # 1. Try to find the result from the last '<<expression=result>>' (GSM8K intermediate/final)
        # This regex looks for <<...=NUMBER>> and captures the number
        expression_result_matches = re.findall(r'<<.*?=(-?\d+(?:\.\d+)?)>>', answer_string)
        if expression_result_matches:
            # Return the last captured numerical result from <<...>>
            return [expression_result_matches[-1]]

        # 2. If no '<<...>>' result found, try to find the final answer after '####' (GSM8K specific)
        # Captures integers and floats
        final_answer_match = re.search(r'####\s*(-?\d+(?:\.\d+)?)', answer_string)
        if final_answer_match:
            return [final_answer_match.group(1)]

        # 3. If neither numerical pattern is found, check for 'true' or 'false' (StrategyQA style)
        if answer_string.strip().lower() == 'true':
            return ['true']
        if answer_string.strip().lower() == 'false':
            return ['false']

        # 4. If none of the above specific patterns, return the original string wrapped in a list
        return [answer_string]
    else:
        # Fallback for unexpected types
        return [str(answer_input)]

# Apply the processing function to the 'answer' column of all dataframes in all_datasets
#print("Processing 'answer' column for all datasets...")
for dataset_name, df in all_datasets.items():
    if 'answer' in df.columns:
        df['answer'] = df['answer'].apply(get_processed_answer_list)
        #print(f"  '{dataset_name}' 'answer' column processed.")


# data CLEANINS
# 2. SPLIT (A) (B) (C)

def split_multi_part_questions(row):
    question_text = row['question']
    original_answers = row['answer'] # This is expected to be a list like ['a', 'b']
    new_rows = []
    parts = re.findall(r'\((\w)\)\s*(.*?)(?=\(\w\)|$)', question_text, re.DOTALL)

    if not parts:
        return [{**row.to_dict(), 'question': question_text, 'answer': original_answers}]

    for letter, q_text in parts:
        is_true = str(letter) in original_answers
        new_row = row.to_dict().copy()
        new_row['question'] = q_text.strip() # Remove leading/trailing whitespace
        new_row['answer'] = ['true'] if is_true else ['false']
        # Remove other columns not relevant for the new simplified question
        if 'qid' in new_row: del new_row['qid']
        if 'term' in new_row: del new_row['term']
        if 'description' in new_row: del new_row['description']
        if 'facts' in new_row: del new_row['facts']
        if 'decomposition' in new_row: del new_row['decomposition']
        if 'evidence' in new_row: del new_row['evidence']

        new_rows.append(new_row)

    return new_rows

#print("The function `split_multi_part_questions` has been defined.")



# data cleaning and processing
###################
# 2 CONTINUES


# Get the maqa_MAQA_commonsense_reasoning DataFrame
maqa_commonsense_df = all_datasets['maqa_MAQA_commonsense_reasoning']

# Apply the function to each row and collect the results
expanded_rows = []
for index, row in maqa_commonsense_df.iterrows():
    expanded_rows.extend(split_multi_part_questions(row))

# Create a new DataFrame from the expanded rows
expanded_maqa_commonsense_df = pd.DataFrame(expanded_rows)

# Replace the original DataFrame in all_datasets with the expanded one
all_datasets['maqa_MAQA_commonsense_reasoning'] = expanded_maqa_commonsense_df


#data processing
# 3
# 18.0 -> 18 ROUND

def round_answers_to_integers(answer_list):
    """
    Rounds numerical strings in a list of answers to integer strings.
    Non-numerical strings are left unchanged.
    """
    processed_answers = []
    for item in answer_list:
        try:
            # Try to convert to float first, then to int, then back to string
            # This handles cases like '18.0' or '70000.0'
            processed_answers.append(str(int(float(item))))
        except (ValueError, TypeError):
            # If it's not a number, or cannot be converted, keep it as is
            processed_answers.append(item)
    return processed_answers


#print("Rounding numerical answers to integers for all datasets...")
for dataset_name, df in all_datasets.items():
    if 'answer' in df.columns:
        df['answer'] = df['answer'].apply(round_answers_to_integers)
        #print(f"  '{dataset_name}' 'answer' column rounded.")



#cleansin
#4 all in format ['string]


#print("Re-processing 'answer' column to ensure ['string'] format for all datasets...")
for dataset_name, df in all_datasets.items():
    if 'answer' in df.columns:
        df['answer'] = df['answer'].apply(get_processed_answer_list)
        #print(f"  '{dataset_name}' 'answer' column re-processed to ['string'] format.")


print("\nVerification of 'answer' format for a few samples:")
# Sample and verify the format for a few dataframes

if 'gsm8k_test' in all_datasets:
    #print("\n--- gsm8k_test sample ---")
    sample_df = all_datasets['gsm8k_test'].head(2)
    for index, row in sample_df.iterrows():
        answer_value = row['answer']
        #print(f"  Question: {row['question'][:50]}...")
        #print(f"  Answer: {answer_value}, Type: {type(answer_value)}, Item Type: {type(answer_value[0]) if answer_value else 'N/A'}")

if 'maqa_MAQA_commonsense_reasoning' in all_datasets:
    #print("\n--- maqa_MAQA_commonsense_reasoning sample ---")
    sample_df = all_datasets['maqa_MAQA_commonsense_reasoning'].head(2)
    for index, row in sample_df.iterrows():
        answer_value = row['answer']
        #print(f"  Question: {row['question'][:50]}...")
        #print(f"  Answer: {answer_value}, Type: {type(answer_value)}, Item Type: {type(answer_value[0]) if answer_value else 'N/A'}")

if 'strategyqa_dev' in all_datasets:
    #print("\n--- strategyqa_dev sample ---")
    sample_df = all_datasets['strategyqa_dev'].head(2)
    for index, row in sample_df.iterrows():
        answer_value = row['answer']
        #print(f"  Question: {row['question'][:50]}...")
        #print(f"  Answer: {answer_value}, Type: {type(answer_value)}, Item Type: {type(answer_value[0]) if answer_value else 'N/A'}")


# Function to determine answer type
def get_answer_type(answer_list):
    if all(isinstance(item, str) and item.lower() in ['true', 'false'] for item in answer_list):
        return 'true/false'
    return 'multi_str'

# Add 'answer_type' column to all dataframes in all_datasets
print("Adding 'answer_type' column to all datasets...")
for dataset_name, df in all_datasets.items():
    if 'answer' in df.columns:
        df['answer_type'] = df['answer'].apply(get_answer_type)
        print(f"  '{dataset_name}' 'answer_type' column added.")

print(df)

#################################
# END OF CLEANSING
################################


# SRC/RUN_DATASETS




RESULTS_PATH = ""
BATCH_SIZE = 1




def run_model_on_dataset():
    """
    This function loops over all datasets, their categories and subcategories
    """

    # TODO: Somehow loop over the datasets
    datasets = [[1]]
    results_df = pd.DataFrame()

    model = Model()

    for dataset in datasets:
        for row in dataset:
            dataframe_to_append = model.run_batch_and_compute_confidence(
                dataset_name="dummy",
                categories = ["dummy_categorie"],
                subcategories = ["dummy_subcategorie"],
                questions = ["What is equal to 2 + 2?"],
                right_answers = [["4"]],
                question_types = ["multi_str"]
                )
            results_df = pd.concat([results_df, dataframe_to_append])

    results_df.to_csv(RESULTS_PATH, index=False, sep=";")


if __name__ == "__main__":
    run_model_on_dataset()
    


ImportError: cannot import name 'Qwen3VLForConditionalGeneration' from 'transformers' (/gpfs/helios/home/hendrika/.conda/envs/llama_local/lib/python3.8/site-packages/transformers/__init__.py)

In [3]:
print(all_datasets)

NameError: name 'all_datasets' is not defined